# CS404 – HW2: Local Search for N-Queens (N = 20)

**Name:** Korhan Erdoğdu  
**Date:** 15.11.2025  
**School No:** 30838

**AI assistance note:**  
I used ChatGPT to help me design the code structure and algorithms
based on the lecture pseudocode for hill climbing that has been shared under the week 4 in the slide named; "chapter04a-localsearch-annotated.pdf".  
I wrote, ran, commented and interpreted all code and results myself.


In [9]:
# CS404 - HW2 - Local Search for N-Queens
# This cell corresponds to the "problem" and common utilities.

import random
import time

# Board size
N = 20


In [10]:
# This cell implements the given heuristic and a VALUE() function which is compatible with the hill-climbing pseudocode.

def calculateNumberOfConflicts(board):
    """
    Returns the number of attacking pairs of queens.
    Representation: board[i] = row of queen in column i.

    There are no vertical conflicts because we place exactly one queen per column.
    Conflicts:
      - same row (horizontal)
      - same diagonal (|Δrow| == |Δcol|)
    """
    numberOfConflicts = 0

    for i in range(len(board)):
        for j in range(i + 1, len(board)):
            # Horizontal conflict (same row)
            if board[i] == board[j]:
                numberOfConflicts += 1
            # Diagonal conflict
            elif abs(i - j) == abs(board[i] - board[j]):
                numberOfConflicts += 1

    return numberOfConflicts


def value(board):
    """
    VALUE function used in pseudocode.
    We want to MINIMIZE conflicts, but the hill-climbing pseudocode
    assumes a MAXIMIZATION problem.

    Therefore: VALUE(board) = - (# of conflicts)
    Higher value = fewer conflicts.
    """
    return -calculateNumberOfConflicts(board)


In [11]:
# This cell defines how we create random states and generate successors.
# It is the concrete version of "INITIAL-STATE" and "successor of current" in the slides.

def random_board(n=N):
    """
    Creates a random board for N-Queens.
    board[i] = row index (0..n-1) of the queen in column i.
    """
    return [random.randrange(n) for _ in range(n)]


def generate_neighbors(board):
    """
    Generates all successors of 'board' by moving ONE queen in ONE column
    to any other row in the same column.

    According to our representation, a successor differs in exactly one position.
    Total neighbors = N * (N-1).
    """
    n = len(board)
    neighbors = []

    for col in range(n):
        current_row = board[col]
        for row in range(n):
            if row == current_row:
                continue  # same position, skip
            new_board = list(board)
            new_board[col] = row
            neighbors.append(new_board)

    return neighbors


In [12]:
# This cell implements BASIC HILL CLIMBING based on the slide pseudocode.
#
# Pseudocode reminder:
#   current <- Make-Node(Initial-State[problem])
#   loop do
#       next <- a highest-valued successor of current
#       if VALUE[next] <= VALUE[current] then return current
#       current <- next

def hill_climbing_basic(start_board, max_steps=10000):
    """
    Performs steepest-ascent hill climbing starting from 'start_board'.

    Returns:
        success (bool)              : whether we reached 0 conflicts
        current_board (list[int])   : final board
        steps (int)                 : number of moves taken
    """
    current_board = list(start_board)
    current_value = value(current_board)  # higher is better

    for step in range(max_steps):
        # Goal check: if there are 0 conflicts, we are done
        if calculateNumberOfConflicts(current_board) == 0:
            return True, current_board, step

        neighbors = generate_neighbors(current_board)

        # Find highest-valued successor (best neighbor)
        best_board = None
        best_value = float("-inf")

        for nb in neighbors:
            v = value(nb)
            if v > best_value:
                best_value = v
                best_board = nb

        # If no improvement, stop (local maximum)
        if best_value <= current_value:
            return False, current_board, step

        # Move to the better neighbor
        current_board = best_board
        current_value = best_value

    # If we hit max_steps, treat as failure
    return False, current_board, max_steps


def run_basic_hill_climbing_one_experiment(n=N, max_steps=10000):
    """
    Runs ONE experiment of basic hill climbing:
      - create random initial state
      - call hill_climbing_basic
      - measure elapsed time

    Returns:
        success (bool)
        elapsed_time (float, seconds)
    """
    start_board = random_board(n)
    t0 = time.time()
    success, final_board, steps = hill_climbing_basic(start_board, max_steps=max_steps)
    t1 = time.time()
    return success, t1 - t0


In [13]:
# This cell implements HILL CLIMBING WITH RANDOM RESTARTS.
#
# Idea:
#   Repeat basic hill climbing from k+1 different random initial states
#   (first state + k restarts). If any run finds a solution, we count it as success.

def run_random_restart_hill_climbing_one_experiment(n=N, k=10, max_steps=10000):
    """
    Runs ONE experiment of Hill Climbing with Random Restart.

    Parameters:
        n : board size
        k : number of restarts allowed
        max_steps : max steps per hill-climbing run

    Returns:
        success (bool)       : True if any restart finds a solution
        elapsed_time (float) : total time for this experiment
    """
    t0 = time.time()
    success = False

    # First attempt + k restarts = k + 1 tries in total
    for attempt in range(k + 1):
        start_board = random_board(n)
        success, final_board, steps = hill_climbing_basic(start_board, max_steps=max_steps)
        if success:
            break

    t1 = time.time()
    return success, t1 - t0


In [14]:
# This cell implements STOCHASTIC HILL CLIMBING.
#
# Algorithm idea:
#   current <- random initial state
#   loop:
#       generate all neighbors
#       keep only neighbors with strictly higher VALUE (fewer conflicts)
#       if no improving neighbor: stop (local max)
#       else: choose one IMPROVING neighbor uniformly at random and move there

def run_stochastic_hill_climbing_one_experiment(n=N, max_steps=10000):
    """
    Runs ONE experiment of stochastic hill climbing.

    Returns:
        success (bool)
        elapsed_time (float, seconds)
    """
    current_board = random_board(n)
    current_value = value(current_board)

    t0 = time.time()

    for step in range(max_steps):
        # Goal test
        if calculateNumberOfConflicts(current_board) == 0:
            t1 = time.time()
            return True, t1 - t0

        neighbors = generate_neighbors(current_board)

        # Collect all improving neighbors
        improving_neighbors = []
        for nb in neighbors:
            v = value(nb)
            if v > current_value:   # strictly better
                improving_neighbors.append((nb, v))

        # If none, we are stuck in a local maximum
        if not improving_neighbors:
            t1 = time.time()
            return False, t1 - t0

        # Pick one improving neighbor at random (stochastic choice)
        nb, v = random.choice(improving_neighbors)
        current_board = nb
        current_value = v

    # Hit max_steps -> failure
    t1 = time.time()
    return False, t1 - t0


In [15]:
# This cell defines a helper to run many experiments and compute statistics.

def run_experiments(run_one_func, num_runs=100):
    """
    Runs 'run_one_func' num_runs times.

    run_one_func is expected to return (success, elapsed_time).

    Returns:
        successes (int)
        success_rate (float, in percent)
        avg_time (float, seconds per experiment)
    """
    successes = 0
    total_time = 0.0

    for _ in range(num_runs):
        success, elapsed = run_one_func()
        if success:
            successes += 1
        total_time += elapsed

    success_rate = successes / num_runs * 100.0
    avg_time = total_time / num_runs
    return successes, success_rate, avg_time


In [16]:
# This cell runs 100 experiments for each required algorithm and prints the results.

NUM_RUNS = 100

# a) Basic Hill Climbing
basic_results = run_experiments(run_basic_hill_climbing_one_experiment, num_runs=NUM_RUNS)
print("Basic Hill Climbing (N=20)")
print(f"  Successes     : {basic_results[0]}/{NUM_RUNS}")
print(f"  Success rate  : {basic_results[1]:.1f}%")
print(f"  Avg time/run  : {basic_results[2]:.4f} seconds\n")

# b) Random Restart with k = 10
def rr_k10():
    return run_random_restart_hill_climbing_one_experiment(k=10)

rr10_results = run_experiments(rr_k10, num_runs=NUM_RUNS)
print("Random Restart Hill Climbing (k=10, N=20)")
print(f"  Successes     : {rr10_results[0]}/{NUM_RUNS}")
print(f"  Success rate  : {rr10_results[1]:.1f}%")
print(f"  Avg time/run  : {rr10_results[2]:.4f} seconds\n")

# b) Random Restart with k = 100
def rr_k100():
    return run_random_restart_hill_climbing_one_experiment(k=100)

rr100_results = run_experiments(rr_k100, num_runs=NUM_RUNS)
print("Random Restart Hill Climbing (k=100, N=20)")
print(f"  Successes     : {rr100_results[0]}/{NUM_RUNS}")
print(f"  Success rate  : {rr100_results[1]:.1f}%")
print(f"  Avg time/run  : {rr100_results[2]:.4f} seconds\n")

# c) Stochastic Hill Climbing
stoch_results = run_experiments(run_stochastic_hill_climbing_one_experiment, num_runs=NUM_RUNS)
print("Stochastic Hill Climbing (N=20)")
print(f"  Successes     : {stoch_results[0]}/{NUM_RUNS}")
print(f"  Success rate  : {stoch_results[1]:.1f}%")
print(f"  Avg time/run  : {stoch_results[2]:.4f} seconds")


Basic Hill Climbing (N=20)
  Successes     : 2/100
  Success rate  : 2.0%
  Avg time/run  : 0.0999 seconds

Random Restart Hill Climbing (k=10, N=20)
  Successes     : 23/100
  Success rate  : 23.0%
  Avg time/run  : 1.1205 seconds

Random Restart Hill Climbing (k=100, N=20)
  Successes     : 92/100
  Success rate  : 92.0%
  Avg time/run  : 4.2373 seconds

Stochastic Hill Climbing (N=20)
  Successes     : 1/100
  Success rate  : 1.0%
  Avg time/run  : 0.1798 seconds


## Short Summary (for Table 1 part d)

For N = 20, random-restart hill climbing with k = 100 reached the highest success rate,
but it also had the largest average running time per experiment.
Basic hill climbing was much faster but frequently got stuck in local optima,
while stochastic hill climbing provided a compromise with better success than basic
hill climbing and moderate running time.
